In [ ]:
import dataclasses as dc
from typing import Self, Type, TypeAlias
import splatlog


@dc.dataclass
class Pair:
    @classmethod
    def from_(cls: Type[Self], value: object) -> Self:
        if isinstance(value, cls):
            return value

        field_a, field_b = dc.fields(cls)

        match value:
            case (a, b) | [a, b] | {field_a.name: a, field_b.name: b}:
                if to_a := field_a.metadata.get("convert"):
                    a = to_a(a)

                if to_b := field_b.metadata.get("convert"):
                    b = to_b(b)

                return cls(a, b)  # type: ignore

        raise TypeError(
            "failed to construct `{}` from `{}`: `{!r}".format(
                cls, type(value), value
            )
        )


@dc.dataclass
class VerbosityLevel(Pair):
    verbosity: splatlog.Verbosity = dc.field(
        metadata={"convert": splatlog.to_verbosity}
    )
    level: splatlog.LevelValue = dc.field(
        metadata={"convert": splatlog.to_level_value}
    )


VerbosityLevel.from_([123, 456])

VerbosityLevel(verbosity=123, level=456)